Experiment Tracking

In [1]:
%pip install -q mlflow scikit-learn pandas numpy joblib

In [4]:
import mlflow
import mlflow.sklearn
import numpy as np
import pandas as pd

from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)

X, y = make_classification(
    n_samples=2000,
    n_features=10,
    n_informative=6,
    n_redundant=2,
    weights=[0.7, 0.3],
    random_state=42
)

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

mlflow.set_experiment("customer-churn-experiments")

results = []

for n_estimators, max_depth in [
    (50, 3),
    (100, 5),
    (150, 8)
]:
    with mlflow.start_run() as run:
        model = RandomForestClassifier(
            n_estimators=n_estimators,
            max_depth=max_depth,
            random_state=42
        )

        model.fit(X_train, y_train)
        predictions = model.predict(X_test)

        accuracy = accuracy_score(y_test, predictions)
        precision = precision_score(
            y_test, predictions, zero_division=0
        )
        recall = recall_score(
            y_test, predictions, zero_division=0
        )
        f1 = f1_score(
            y_test, predictions, zero_division=0
        )

        mlflow.log_param("n_estimators", n_estimators)
        mlflow.log_param("max_depth", max_depth)

        mlflow.log_metrics({
            "accuracy": accuracy,
            "precision": precision,
            "recall": recall,
            "f1_score": f1
        })

        results.append({
            "run_id": run.info.run_id,
            "n_estimators": n_estimators,
            "max_depth": max_depth,
            "accuracy": accuracy,
            "f1_score": f1
        })

results_df = pd.DataFrame(results)
results_df.sort_values("f1_score", ascending=False)

,run_id,n_estimators,max_depth,accuracy,f1_score
2,f32aefb113654e9688f9f1fe41fa1c31,150,8,0.9350,0.884956
1,59c769fac273484e9f8ae11d6fe20706,100,5,0.9125,0.841629
0,392143e5f2f143abb9ba973331e09e7b,50,3,0.9100,0.836364


In [5]:
best_run = results_df.sort_values(
    "f1_score", ascending=False
).iloc[0]

print("Best run ID:", best_run["run_id"])
print("Best F1-score:", best_run["f1_score"])
print("Best parameters:")
print("n_estimators:", best_run["n_estimators"])
print("max_depth:", best_run["max_depth"])

Best run ID: f32aefb113654e9688f9f1fe41fa1c31
Best F1-score: 0.8849557522123894
Best parameters:
n_estimators: 150
max_depth: 8


Register a model with MLflow

In [6]:
import mlflow
import mlflow.sklearn
from mlflow import MlflowClient

mlflow.set_tracking_uri("sqlite:///mlflow.db")

mlflow.set_experiment("customer-churn-registry")

print("Tracking URI:", mlflow.get_tracking_uri())

2026/10/25 15:00:05 INFO mlflow.tracking.fluent: Experiment with name 'customer-churn-registry' does not exist. Creating a new experiment.


Tracking URI: sqlite:///mlflow.db


In [11]:
#Train, log, and register a model

with mlflow.start_run() as run:
    model = RandomForestClassifier(
        n_estimators=100,
        max_depth=5,
        random_state=42
    )

    model.fit(X_train, y_train)

    predictions = model.predict(X_test)
    score = f1_score(y_test, predictions)

    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("max_depth", 5)
    mlflow.log_metric("f1_score", score)

print("F1-score:", score)

F1-score: 0.8416289592760181


In [12]:
#Inspect the registered model

client = MlflowClient()

for mv in client.search_model_versions(
    "name = 'CustomerChurnModel'"
):
    print(
        "Version:", mv.version,
        "| Run ID:", mv.run_id,
        "| Status:", mv.status
    )

In [ ]:
#Load the registered version

registered_uri = "models:/CustomerChurnModel/1"

registered_model = mlflow.sklearn.load_model(
    registered_uri
)

print(registered_model.predict(X_test[:5]))

Deploy a model using FastAPI in Colab

In [14]:
%pip install -q fastapi uvicorn joblib requests

In [17]:
import joblib

joblib.dump(file_name,"churn_model.joblib")
print("Model saved successfully")

NameError: name 'file_name' is not defined

In [20]:
#Create the API

%%writefile app.py

from fastapi import FastAPI
from pydantic import BaseModel, Field
import joblib

app = FastAPI(title="Customer Churn Prediction API")

model = joblib.load("churn_model.joblib")

class PredictionRequest(BaseModel):
    features: list[float] = Field(
        ...,
        min_length=10,
        max_length=10
    )

@app.get("/")
def home():
    return {"message": "Churn Prediction API is running"}

@app.post("/predict")
def predict(request: PredictionRequest):
    prediction = int(
        model.predict([request.features])[0]
    )

    probability = float(
        model.predict_proba([request.features])[0][1]
    )

    return {
        "prediction": prediction,
        "churn_probability": probability
    }

Overwriting app.py


In [19]:
%%writefile app.py

from fastapi import FastAPI
from pydantic import BaseModel, Field
import joblib

app = FastAPI(title="Customer Churn Prediction API")

model = joblib.load("churn_model.joblib")

class PredictionRequest(BaseModel):
    features: list[float] = Field(
        ...,
        min_length=10,
        max_length=10
    )

@app.get("/")
def home():
    return {"message": "Churn Prediction API is running"}

@app.post("/predict")
def predict(request: PredictionRequest):
    prediction = int(
        model.predict([request.features])[0]
    )

    probability = float(
        model.predict_proba([request.features])[0][1]
    )

    return {
        "prediction": prediction,
        "churn_probability": probability
    }

Overwriting app.py


In [21]:
#Start the API

import subprocess
import time

server = subprocess.Popen([
    "uvicorn", "app:app",
    "--host", "127.0.0.1",
    "--port", "8000"
])

time.sleep(3)
print("API process started")

API process started


In [ ]:
#Test the API

import requests

response = requests.post(
    "http://127.0.0.1:8000/predict",
    json={"features": X_test[0].tolist()}
)

print("Status:", response.status_code)
print(response.json())

Create a GitHub Actions CI pipeline

In [1]:
#Create a minimal project test

%%writefile test_model.py

from sklearn.datasets import make_classification
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score

def test_model_quality():
    X, y = make_classification(
        n_samples=500,
        n_features=10,
        random_state=42
    )

    X_train, X_test, y_train, y_test = train_test_split(
        X, y,
        test_size=0.2,
        random_state=42
    )

    model = RandomForestClassifier(
        n_estimators=50,
        random_state=42
    )

    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    score = f1_score(y_test, predictions)

    assert 0.0 <= score <= 1.0
    assert len(predictions) == len(y_test)

Writing test_model.py


In [2]:
#run the test locally
%pip install -q pytest
!pytest -q test_model.py

.                                                                        [100%]
1 passed in 2.45s


In [3]:
#Create the workflow

import os

os.makedirs(".github/workflows", exist_ok=True)

In [4]:
%%writefile .github/workflows/ml-ci.yml

name: ML CI

on:
  push:
    branches: [main]
  pull_request:
    branches: [main]

jobs:
  test:
    runs-on: ubuntu-latest

    steps:
      - name: Checkout code
        uses: actions/checkout@v4

      - name: Set up Python
        uses: actions/setup-python@v5
        with:
          python-version: "3.11"

      - name: Install dependencies
        run: |
          python -m pip install --upgrade pip
          pip install pytest scikit-learn

      - name: Run tests
        run: pytest -q

Writing .github/workflows/ml-ci.yml
